In [150]:
%pip install scikit-learn

Python(80772) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


Note: you may need to restart the kernel to use updated packages.


In [151]:
from pathlib import Path
import pandas as pd
import numpy as np
import re

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Import Data

In [152]:
#creates Path object pointing to folder
data_path = Path("/Users/kierachiu/Documents/IDX Exchange/IDX-Exchange-Data-Science/data/csv")

#creates empty list to store wanted files
files = []

for file in data_path.glob("CRMLSSold*.csv"): #looking for specific format
    if re.fullmatch(r"CRMLSSold\d{6}(?:_filled)?.csv", file.name):
        files.append(file)

#sort files by date
files = sorted(files)

print(f"Found {len(files)} files")

for file in files:
    print(file.name)

Found 28 files
CRMLSSold202401_filled.csv
CRMLSSold202402.csv
CRMLSSold202403_filled.csv
CRMLSSold202404_filled.csv
CRMLSSold202405_filled.csv
CRMLSSold202406_filled.csv
CRMLSSold202407_filled.csv
CRMLSSold202408.csv
CRMLSSold202409.csv
CRMLSSold202410.csv
CRMLSSold202411.csv
CRMLSSold202412.csv
CRMLSSold202501_filled.csv
CRMLSSold202502.csv
CRMLSSold202503.csv
CRMLSSold202504.csv
CRMLSSold202505.csv
CRMLSSold202506.csv
CRMLSSold202507.csv
CRMLSSold202508.csv
CRMLSSold202509.csv
CRMLSSold202510.csv
CRMLSSold202511.csv
CRMLSSold202512.csv
CRMLSSold202601.csv
CRMLSSold202602.csv
CRMLSSold202603.csv
CRMLSSold202604.csv


In [153]:
# stores all modified CSVs in a list
dfs = []

for file in files:
    df_temp = pd.read_csv(file, low_memory=False)

    # create column to track original CSV file name
    df_temp["source_file"] = file.name

    # extract YYYYMM from filename
    month_match = re.search(r"\d{6}", file.name)

    if month_match:
        # create new column and add data_month 
        df_temp["data_month"] = month_match.group()

    dfs.append(df_temp)

df = pd.concat(dfs, ignore_index=True) #discards old indexes after concatenating

# Filter for only Residential, SingleFamilyResidence 
df_model = df[(df["PropertyType"] == "Residential") & (df["PropertySubType"] == "SingleFamilyResidence")].copy()

# Preserve the initial modeling dataset
df_raw_model = df_model.copy()

# Create an empty list to store each cleaning operation
cleaning_log = []
# Store original number of rows
initial_rows = len(df_model)

print(f"Starting rows: {initial_rows:,}")
print(f"Starting columns: {df_model.shape[1]:,}")

print("Original rows:", len(df))
print("Single-family residential rows:", len(df_model))

Starting rows: 309,735
Starting columns: 86
Original rows: 615707
Single-family residential rows: 309735


In [154]:
# records info abt cleaning operations 
def log_cleaning_step(step_name, before, after):
    
    removed = before - after
    percent_removed = (removed / before * 100 if before > 0 else 0)

    # store results in the cleaning log
    cleaning_log.append({
        "step": step_name,
        "rows_before": before,
        "rows_removed": removed,
        "rows_after": after,
        "percent_removed": round(percent_removed, 2)})

    print(f"\n{step_name}")
    print(f"Rows before: {before:,}")
    print(f"Rows removed: {removed:,}")
    print(f"Rows after: {after:,}")
    print(f"Percentage removed: {percent_removed:.2f}%")

# Cleaning Operations

In [155]:
# convert ClosePrice into a number, replace with NaN if can't convert value to number
df_model["ClosePrice"] = pd.to_numeric(df_model["ClosePrice"], errors="coerce")

# count rows before removing missing prices
before = len(df_model)

# keep observations where ClosePrice is not missing
df_model = df_model[df_model["ClosePrice"].notna()].copy()

# record how many rows were removed
log_cleaning_step("Remove missing ClosePrice", before, len(df_model))


Remove missing ClosePrice
Rows before: 309,735
Rows removed: 2
Rows after: 309,733
Percentage removed: 0.00%


In [156]:
before = len(df_model)

# Keep only properties sold for a positive price
df_model = df_model[df_model["ClosePrice"] > 0].copy()

log_cleaning_step("Remove ClosePrice <= 0", before, len(df_model))


Remove ClosePrice <= 0
Rows before: 309,733
Rows removed: 1
Rows after: 309,732
Percentage removed: 0.00%


In [157]:
# exclude import-tracking columns from comparison
tracking_columns = ["source_file", "data_month"]
comparison_columns = [col for col in df_model.columns if col not in tracking_columns]

before = len(df_model)

# Remove duplicated source records
df_model = df_model.drop_duplicates(subset=comparison_columns, keep="first").copy()

log_cleaning_step("Remove exact duplicate records", before, len(df_model))


Remove exact duplicate records
Rows before: 309,732
Rows removed: 29
Rows after: 309,703
Percentage removed: 0.01%


In [158]:
# Identify rows with repeated ListingKeys
duplicate_mask = (
    df_model["ListingKey"].notna() &
    df_model["ListingKey"].duplicated(keep=False)
)

duplicate_listings = df_model.loc[duplicate_mask,
    ["ListingKey", "CloseDate", "ClosePrice", 
     "source_file", "data_month"]].sort_values("ListingKey")

print("Rows with repeated ListingKeys:", len(duplicate_listings))

display(duplicate_listings.head(10))

Rows with repeated ListingKeys: 464


,ListingKey,CloseDate,ClosePrice,source_file,data_month
231924,1031992525,2024-10-18,459900.0,CRMLSSold202410.csv,202410
136532,1031992525,2024-06-21,415740.0,CRMLSSold202406_filled.csv,202406
208611,1037464932,2024-09-06,665000.0,CRMLSSold202409.csv,202409
112046,1037464932,2024-05-25,650000.0,CRMLSSold202405_filled.csv,202405
231725,1046689006,2024-10-29,825000.0,CRMLSSold202410.csv,202410
135852,1046689006,2024-06-30,825000.0,CRMLSSold202406_filled.csv,202406
186923,1048638478,2024-08-21,6550000.0,CRMLSSold202408.csv,202408
135638,1048638478,2024-06-25,6600000.0,CRMLSSold202406_filled.csv,202406
186886,1052182218,2024-08-20,355000.0,CRMLSSold202408.csv,202408
82994,1052182218,2024-04-26,355000.0,CRMLSSold202404_filled.csv,202404


In [159]:
# Ensure CloseDate is comparable as a date
df_model["CloseDate"] = pd.to_datetime(df_model["CloseDate"], errors="coerce")

transaction_keys = [
    "ListingKey",
    "CloseDate",
    "ClosePrice"]

# Only deduplicate rows with complete transaction identifiers
has_keys = df_model[transaction_keys].notna().all(axis=1)

before = len(df_model)

duplicate_transactions = (has_keys & df_model.duplicated(subset=transaction_keys,keep="first"))

df_model = df_model.loc[~duplicate_transactions].copy()

log_cleaning_step("Remove repeated listing transactions", before, len(df_model))


Remove repeated listing transactions
Rows before: 309,703
Rows removed: 0
Rows after: 309,703
Percentage removed: 0.00%


In [160]:
# convert ListingContractDate into datetime format
df_model["ListingContractDate"] = pd.to_datetime(df_model["ListingContractDate"], errors="coerce")

# Examine missing dates
print("Missing listing contract dates:", df_model["ListingContractDate"].isna().sum())

print("Missing closing dates:", df_model["CloseDate"].isna().sum())

invalid_date_mask = (
    df_model["CloseDate"].notna() &
    df_model["ListingContractDate"].notna() &
    (df_model["CloseDate"] < df_model["ListingContractDate"]))

invalid_dates = df_model.loc[invalid_date_mask]

print("Transactions with impossible date sequences:", len(invalid_dates))

display(invalid_dates[["ListingKey", "ListingContractDate", "CloseDate", "ClosePrice"]].head(20))

Missing listing contract dates: 0
Missing closing dates: 0
Transactions with impossible date sequences: 44


,ListingKey,ListingContractDate,CloseDate,ClosePrice
52,1059827487,2024-01-31,2024-01-30,2160000.0
149,1059357897,2024-01-26,2024-01-25,2705000.0
38156,1063548623,2024-03-21,2024-03-20,1650000.0
112559,1076675910,2024-06-20,2024-06-13,715000.0
163391,1079658093,2024-08-14,2024-08-13,5600000.0
191650,1080048344,2024-10-02,2024-09-19,1305000.0
191951,1079998553,2024-09-30,2024-09-05,995000.0
192787,1079658804,2024-09-28,2024-09-24,1275000.0
209258,1089974300,2024-10-11,2024-10-10,5200000.0
218308,1082234342,2024-10-17,2024-10-09,1903500.0


In [161]:
before = len(df_model)

df_model = df_model.loc[~invalid_date_mask].copy()

log_cleaning_step("Remove closing dates before listing dates", before, len(df_model))


Remove closing dates before listing dates
Rows before: 309,703
Rows removed: 44
Rows after: 309,659
Percentage removed: 0.01%


In [162]:
before = len(df_model)

# Identify known invalid living areas
invalid_living_area = (
    df_model["LivingArea"].notna() &
    (df_model["LivingArea"] <= 0))

print("Properties with LivingArea <= 0:", invalid_living_area.sum())

# Keep all rows except those with invalid living area
df_model = df_model.loc[~invalid_living_area].copy()

log_cleaning_step("Remove nonpositive LivingArea", before, len(df_model))

Properties with LivingArea <= 0: 126

Remove nonpositive LivingArea
Rows before: 309,659
Rows removed: 126
Rows after: 309,533
Percentage removed: 0.04%


In [163]:
count_columns = [
    "BedroomsTotal",
    "BathroomsTotalInteger"]

for col in count_columns:

    before_rows = len(df_model)

    invalid_mask = df_model[col] < 0
    invalid_count = invalid_mask.sum()

    # Replace negative counts with missing values
    df_model.loc[invalid_mask, col] = np.nan

    print(f"\n{col}")
    print(f"Invalid values replaced with NaN: {invalid_count:,}")

    log_cleaning_step(f"Replace negative {col} with NaN", before_rows, len(df_model))


BedroomsTotal
Invalid values replaced with NaN: 0

Replace negative BedroomsTotal with NaN
Rows before: 309,533
Rows removed: 0
Rows after: 309,533
Percentage removed: 0.00%

BathroomsTotalInteger
Invalid values replaced with NaN: 0

Replace negative BathroomsTotalInteger with NaN
Rows before: 309,533
Rows removed: 0
Rows after: 309,533
Percentage removed: 0.00%


In [164]:
before = len(df_model)

invalid_lot_mask = (df_model["LotSizeSquareFeet"] <= 0)

invalid_lot_count = invalid_lot_mask.sum()

df_model.loc[invalid_lot_mask, "LotSizeSquareFeet"] = np.nan

print("Invalid lot-size values replaced:", invalid_lot_count)

log_cleaning_step("Replace nonpositive lot sizes with NaN", before, len(df_model))

Invalid lot-size values replaced: 162

Replace nonpositive lot sizes with NaN
Rows before: 309,533
Rows removed: 0
Rows after: 309,533
Percentage removed: 0.00%


In [182]:
# Check whether CSV month matches actual closing month
date_check = df_model[["data_month", "CloseDate"]].copy()

date_check["CloseDate"] = pd.to_datetime(
    date_check["CloseDate"],
    errors="coerce"
)

date_check["closing_month"] = (
    date_check["CloseDate"].dt.strftime("%Y%m")
)

date_check["month_matches"] = (
    date_check["data_month"].astype(str)
    == date_check["closing_month"]
)

print("Missing closing dates:",
      date_check["CloseDate"].isna().sum())

print("Rows where months differ:",
      (~date_check["month_matches"]).sum())

print("\nMismatch breakdown:")
print(
    date_check.loc[
        ~date_check["month_matches"]
    ].groupby(["data_month", "closing_month"], dropna=False)
     .size()
     .sort_values(ascending=False)
     .head(20)
)

Missing closing dates: 0
Rows where months differ: 0

Mismatch breakdown:
Series([], dtype: int64)


In [165]:
# Convert the list of cleaning operations into a DataFrame
cleaning_report = pd.DataFrame(cleaning_log)

# Display the complete audit
display(cleaning_report)

# Calculate overall retention statistics
final_rows = len(df_model)
total_removed = initial_rows - final_rows

overall_removed_percent = (
    total_removed / initial_rows * 100
    if initial_rows > 0 else 0)

overall_retained_percent = (
    final_rows / initial_rows * 100
    if initial_rows > 0 else 0)


print("=" * 45)
print("FINAL DATA CLEANING SUMMARY")
print("=" * 45)

print(f"Starting observations: {initial_rows:,}")
print(f"Final observations: {final_rows:,}")
print(f"Total observations removed: {total_removed:,}")
print(f"Percentage removed: {overall_removed_percent:.2f}%")
print(f"Percentage retained: {overall_retained_percent:.2f}%")

,step,rows_before,rows_removed,rows_after,percent_removed
0,Remove missing ClosePrice,309735,2,309733,0.00
1,Remove ClosePrice <= 0,309733,1,309732,0.00
2,Remove exact duplicate records,309732,29,309703,0.01
3,Remove repeated listing transactions,309703,0,309703,0.00
4,Remove closing dates before listing dates,309703,44,309659,0.01
5,Remove nonpositive LivingArea,309659,126,309533,0.04
6,Replace negative BedroomsTotal with NaN,309533,0,309533,0.00
7,Replace negative BathroomsTotalInteger with NaN,309533,0,309533,0.00
8,Replace nonpositive lot sizes with NaN,309533,0,309533,0.00


FINAL DATA CLEANING SUMMARY
Starting observations: 309,735
Final observations: 309,533
Total observations removed: 202
Percentage removed: 0.07%
Percentage retained: 99.93%


## Determining numeric features

In [166]:
# Identify all numerical features
available_numeric_features = df_model.select_dtypes(include=["number"]).columns.tolist()

# Exclude target variable and leakage features
exclude_columns = [
    # Target variable
    "ClosePrice",

    # Listing prices
    "ListPrice",
    "OriginalListPrice",
    "ListPriceLow",
    "CurrentPrice",
    "PreviousListPrice",

    # Days on Market
    "DaysOnMarket",
    "CumulativeDaysOnMarket",
    "DaysOnMarketReplication",
    "DaysOnMarketReplicationDate",
    "DaysOnMarketReplicationIncreasingYN",

    # Price changes
    "PriceChangeTimestamp",
    "MajorChangeType",
    "MajorChangeTimestamp",

    # Closing and settlement
    "CloseDate",
    "EstimatedCloseDate",
    "BuyerFinancing",
    "Concessions",
    "ConcessionsAmount",
    "ConcessionsComments",

    # Sale contracts
    "PurchaseContractDate",
    "ContractStatusChangeDate",
    "PendingTimestamp",
    "ContingentDate",
]

# Keep only features that are not excluded
available_numeric_features = [
    col for col in available_numeric_features
    if col not in exclude_columns
]

# Display results
print("Number of numerical features:", len(available_numeric_features))

print("\nNumerical features:")
print(available_numeric_features)

Number of numerical features: 27

Numerical features:
['ListingKey', 'Latitude', 'Longitude', 'LivingArea', 'FireplacesTotal', 'AboveGradeFinishedArea', 'ListingKeyNumeric', 'TaxAnnualAmount', 'ParkingTotal', 'LotSizeAcres', 'YearBuilt', 'StreetNumberNumeric', 'BathroomsTotalInteger', 'BuyerAgencyCompensation', 'TaxYear', 'BuildingAreaTotal', 'BedroomsTotal', 'ElementarySchoolDistrict', 'BelowGradeFinishedArea', 'CoveredSpaces', 'Stories', 'LotSizeArea', 'MainLevelBedrooms', 'GarageSpaces', 'AssociationFee', 'LotSizeSquareFeet', 'MiddleOrJuniorSchoolDistrict']


In [167]:
missing_summary = pd.DataFrame({
    "Feature": available_numeric_features,
    "Missing_Count": [
        df_model[col].isna().sum() for col in available_numeric_features
    ],
    "Missing_Percentage": [
        df_model[col].isna().mean() * 100
        for col in available_numeric_features]})

# sort by highest missingness
missing_summary = missing_summary.sort_values(
    by="Missing_Percentage",
    ascending=False).reset_index(drop=True)

missing_summary["Missing_Percentage"] = (missing_summary["Missing_Percentage"].round(2))

# print missingness table
print("\nNumeric Feature Missingness Summary:")
display(missing_summary)


Numeric Feature Missingness Summary:


,Feature,Missing_Count,Missing_Percentage
0,MiddleOrJuniorSchoolDistrict,309533,100.00
1,TaxYear,309533,100.00
2,CoveredSpaces,309533,100.00
3,FireplacesTotal,309533,100.00
4,AboveGradeFinishedArea,309533,100.00
5,ElementarySchoolDistrict,309533,100.00
6,TaxAnnualAmount,309533,100.00
7,BelowGradeFinishedArea,307630,99.39
8,BuildingAreaTotal,289988,93.69
9,BuyerAgencyCompensation,267302,86.36


In [168]:
# Drop features with extreme missingness or low usefulness
columns_to_drop = [
    "MiddleOrJuniorSchoolDistrict",
    "ElementarySchoolDistrict",
    "TaxYear",
    "TaxAnnualAmount",
    "BuyerAgencyCompensation",
    "MainLevelBedrooms",
    "ListingKey",
    "AssociationFee",
    'FireplacesTotal',          
    'AboveGradeFinishedArea',
    'CoveredSpaces']

# Record number of columns before cleaning
before = len(available_numeric_features)

# Remove excluded features from numeric feature list
available_numeric_features = [
    col for col in available_numeric_features
    if col not in columns_to_drop]

# Record number of columns after cleaning
after = len(available_numeric_features)

# Log results using your existing function
log_cleaning_step("Remove high-missingness and identifier features (columns)", before, after)

print("\nRemaining numeric features:")
print(available_numeric_features)



Remove high-missingness and identifier features (columns)
Rows before: 27
Rows removed: 11
Rows after: 16
Percentage removed: 40.74%

Remaining numeric features:
['Latitude', 'Longitude', 'LivingArea', 'ListingKeyNumeric', 'ParkingTotal', 'LotSizeAcres', 'YearBuilt', 'StreetNumberNumeric', 'BathroomsTotalInteger', 'BuildingAreaTotal', 'BedroomsTotal', 'BelowGradeFinishedArea', 'Stories', 'LotSizeArea', 'GarageSpaces', 'LotSizeSquareFeet']


## Determining categorical features

In [137]:
# Identify categorical columns
available_categorical_features = df_model.select_dtypes(include=["object", "category", "bool", "string"]).columns.tolist()

print("Categorical features:")
print(available_categorical_features)

Categorical features:
['Flooring', 'ViewYN', 'WaterfrontYN', 'BasementYN', 'PoolPrivateYN', 'ListAgentEmail', 'ListAgentFirstName', 'ListAgentLastName', 'UnparsedAddress', 'PropertyType', 'ListOfficeName', 'BuyerOfficeName', 'CoListOfficeName', 'ListAgentFullName', 'CoListAgentFirstName', 'CoListAgentLastName', 'BuyerAgentMlsId', 'BuyerAgentFirstName', 'BuyerAgentLastName', 'AssociationFeeFrequency', 'MLSAreaMajor', 'CountyOrParish', 'MlsStatus', 'ElementarySchool', 'AttachedGarageYN', 'BuilderName', 'PropertySubType', 'SubdivisionName', 'BuyerOfficeAOR', 'BuyerAgencyCompensationType', 'ListingId', 'City', 'ContractStatusChangeDate', 'CoBuyerAgentFirstName', 'PurchaseContractDate', 'BusinessType', 'StateOrProvince', 'MiddleOrJuniorSchool', 'FireplaceYN', 'HighSchool', 'Levels', 'LotSizeDimensions', 'NewConstructionYN', 'HighSchoolDistrict', 'PostalCode', 'latfilled', 'lonfilled', 'source_file', 'data_month', 'BuyerAgentAOR', 'ListAgentAOR', 'OriginatingSystemName', 'OriginatingSystemSu

In [138]:
# Make a separate copy for categorical feature inspection
categorical_review = df_model[available_categorical_features].copy()

# Build categorical feature quality report
categorical_quality = pd.DataFrame({
    "feature": available_categorical_features,

    "dtype": [
        str(df_model[col].dtype)
        for col in available_categorical_features
    ],

    "missing_count": [
        categorical_review[col].isna().sum()
        for col in available_categorical_features
    ],

    "missing_percent": [
        categorical_review[col].isna().mean() * 100
        for col in available_categorical_features
    ],

    "unique_values": [
        categorical_review[col].nunique()
        for col in available_categorical_features
    ],

    "mode": [
        categorical_review[col].mode().iloc[0]
        if not categorical_review[col].mode().empty
        else None
        for col in available_categorical_features
    ]})

# Round missingness percentage
categorical_quality["missing_percent"] = (categorical_quality["missing_percent"].round(2))

# Display sorted report
display(categorical_quality.sort_values("missing_percent"))

,feature,dtype,missing_count,missing_percent,unique_values,mode
26,PropertySubType,str,0,0.00,1,SingleFamilyResidence
30,ListingId,str,0,0.00,309299,CV25004341
36,StateOrProvince,str,0,0.00,12,CA
22,MlsStatus,str,0,0.00,1,Closed
21,CountyOrParish,str,0,0.00,63,Los Angeles
44,PostalCode,str,1,0.00,3190,92253
9,PropertyType,str,0,0.00,1,Residential
10,ListOfficeName,str,0,0.00,16694,Compass
48,data_month,str,0,0.00,28,202405
47,source_file,str,0,0.00,28,CRMLSSold202405_filled.csv


In [131]:
# Categorical features to investigate
categorical_candidates = [
    "PostalCode",
    "CountyOrParish",
    "City",
    "HighSchoolDistrict",
    "ArchitecturalStyle",
    "CoolingYN",
    "HeatingYN",
    "NewConstructionYN"
]

# Keep available categorical candidates
categorical_candidates = [
    col for col in categorical_candidates
    if col in df_model.columns
]

# Examine missingness and cardinality
categorical_selection_report = pd.DataFrame({
    "feature": categorical_candidates,
    "missing_percent": [
        df_model[col].isna().mean() * 100
        for col in categorical_candidates
    ],
    "unique_values": [
        df_model[col].nunique()
        for col in categorical_candidates
    ]})

display(categorical_selection_report.sort_values("missing_percent"))

,feature,missing_percent,unique_values
1,CountyOrParish,0.000000,63
0,PostalCode,0.000323,3190
2,City,0.078505,1105
4,NewConstructionYN,7.258354,2
3,HighSchoolDistrict,26.004012,443


## Leakage analysis

In [133]:
# Known leakage-prone features
leakage_features = [
    "ListPrice",
    "OriginalListPrice",
    "ListPriceLow",
    "CurrentPrice",
    "ClosePrice",
    "DaysOnMarket",
    "CumulativeDaysOnMarket",
    "CloseDate",
    "PurchaseContractDate",
    "BuyerFinancing",
    "Concessions",
    "ConcessionsAmount",
    "PriceChangeTimestamp"
]

# Check which leakage features exist in your dataset
existing_leakage_features = [
    col for col in leakage_features
    if col in df_model.columns
]

print("Potential leakage features found:")
print(existing_leakage_features)

Potential leakage features found:
['ListPrice', 'OriginalListPrice', 'ClosePrice', 'DaysOnMarket', 'CloseDate', 'PurchaseContractDate']


In [173]:
# Use a separate variable for correlation analysis
correlation_numeric_columns = df_model.select_dtypes(
    include=["number"]
).columns.tolist()

correlation_with_price = (
    df_model[correlation_numeric_columns]
    .corr()["ClosePrice"]
    .drop("ClosePrice")
    .sort_values(ascending=False)
)

display(
    correlation_with_price.to_frame(
        name="correlation_with_ClosePrice"))

,correlation_with_ClosePrice
BuildingAreaTotal,0.639627
ListPrice,0.248288
LivingArea,0.152893
BathroomsTotalInteger,0.130477
BedroomsTotal,0.084104
OriginalListPrice,0.053859
AssociationFee,0.048487
Stories,0.039669
BuyerAgencyCompensation,0.033574
BelowGradeFinishedArea,0.025680


In [135]:
# Identify features with very high absolute correlation
high_correlation_features = (
    correlation_with_price[
        correlation_with_price.abs() >= 0.95])

print("Features with suspiciously high correlation:")

display(
    high_correlation_features.to_frame(
        name="correlation_with_ClosePrice"))

Features with suspiciously high correlation:


,correlation_with_ClosePrice


In [136]:
# Build leakage audit report
leakage_audit = pd.DataFrame({
    "feature": [
        col for col in df_model.columns
        if col != "ClosePrice"
    ]
})

# Flag known timing-related leakage
leakage_audit["timing_risk"] = (
    leakage_audit["feature"]
    .isin(existing_leakage_features)
)

# Add numerical correlations
leakage_audit["correlation_with_ClosePrice"] = (
    leakage_audit["feature"]
    .map(correlation_with_price)
)

# Flag high absolute correlations
leakage_audit["high_correlation_risk"] = (
    leakage_audit["correlation_with_ClosePrice"]
    .abs()
    .ge(0.95)
)

# Flag features requiring review
leakage_audit["review_required"] = (
    leakage_audit["timing_risk"] |
    leakage_audit["high_correlation_risk"]
)

# Display highest-risk features first
display(
    leakage_audit.sort_values(
        ["review_required", "high_correlation_risk"],
        ascending=False
    )
)

,feature,timing_risk,correlation_with_ClosePrice,high_correlation_risk,review_required
5,OriginalListPrice,True,0.053859,False,True
8,CloseDate,True,NaN,False,True
16,ListPrice,True,0.248288,False,True
17,DaysOnMarket,True,0.010426,False,True
56,PurchaseContractDate,True,NaN,False,True
...,...,...,...,...,...
80,data_month,False,NaN,False,False
81,BuyerAgentAOR,False,NaN,False,False
82,ListAgentAOR,False,NaN,False,False
83,OriginatingSystemName,False,NaN,False,False


In [66]:
# recalculate feature quality
feature_review = df_model[available_numeric_features].copy()

for col in available_numeric_features:
    feature_review[col] = pd.to_numeric(
        feature_review[col],
        errors="coerce"
    )

feature_quality = pd.DataFrame({
    "feature": available_numeric_features,
    "missing_count": feature_review.isna().sum().values,
    "missing_percent": (
        feature_review.isna().mean().values * 100
    ).round(2),
    "unique_values": feature_review.nunique().values,
    "median": feature_review.median().values
})

display(
    feature_quality.sort_values("missing_percent")
)

,feature,missing_count,missing_percent,unique_values,median
3,BedroomsTotal,0,0.00,23,3.000000
4,BathroomsTotalInteger,50,0.02,30,2.000000
0,LivingArea,166,0.05,7127,1804.000000
5,YearBuilt,225,0.07,165,1976.000000
7,ParkingTotal,348,0.11,190,2.000000
8,Latitude,2712,0.88,281676,34.090326
9,Longitude,2712,0.88,286910,-118.040031
1,LotSizeSquareFeet,5317,1.72,29850,7245.000000
2,LotSizeAcres,5344,1.73,13561,0.166300
6,GarageSpaces,11510,3.72,46,2.000000


# Train/Test Feature Setup 

In [178]:
# Chronological train/validation/test split

months = sorted(df_model["data_month"].dropna().unique())

# Choose the training window length
X = 4  # Try 2, 3, 4, 5, etc.

# Most recent complete month = test
test_month = months[-1]

# Second-most-recent complete month = validation
validation_month = months[-2]

# X months immediately before validation = training
train_months = months[-(X + 2):-2]

train_df = df_model[
    df_model["data_month"].isin(train_months)
].copy()

val_df = df_model[
    df_model["data_month"] == validation_month
].copy()

test_df = df_model[
    df_model["data_month"] == test_month
].copy()

print("Training months:", train_months)
print("Validation month:", validation_month)
print("Test month:", test_month)

print("Training rows:", len(train_df))
print("Validation rows:", len(val_df))
print("Testing rows:", len(test_df))

Training months: ['202511', '202512', '202601', '202602']
Validation month: 202603
Test month: 202604
Training rows: 36626
Validation rows: 11560
Testing rows: 12482


In [183]:
target = "ClosePrice"

lower_threshold = train_df[target].quantile(0.005)
upper_threshold = train_df[target].quantile(0.995)

print(f"0.5th percentile: ${lower_threshold:,.2f}")
print(f"99.5th percentile: ${upper_threshold:,.2f}")

0.5th percentile: $180,000.00
99.5th percentile: $8,999,375.00


In [184]:
# Store original training size
original_train_size = len(train_df)

# Remove training rows outside the thresholds
train_df = train_df[
    train_df[target].between(
        lower_threshold,
        upper_threshold,
        inclusive="both"
    )
].copy()

# Report changes
removed_rows = original_train_size - len(train_df)

print("Original training rows:", original_train_size)
print("Remaining training rows:", len(train_df))
print("Removed outliers:", removed_rows)

print(f"Percentage removed: {removed_rows / original_train_size * 100:.2f}%")

Original training rows: 36626
Remaining training rows: 36264
Removed outliers: 362
Percentage removed: 0.99%


In [185]:
print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

assert val_df["ClosePrice"].notna().all()
assert test_df["ClosePrice"].notna().all()

print("Training-only outlier filtering completed.")

Train: (36264, 89)
Validation: (11560, 89)
Test: (12482, 89)
Training-only outlier filtering completed.


In [179]:


def add_temporal_features(data):
    data = data.copy()

    # Get the year and month from data_month (YYYYMM)
    dates = pd.to_datetime(
        data["data_month"].astype(str),
        format="%Y%m",
        errors="coerce"
    )

    month = dates.dt.month
    year = dates.dt.year

    # Capture seasonality
    data["month_sin"] = np.sin(2 * np.pi * month / 12)
    data["month_cos"] = np.cos(2 * np.pi * month / 12)

    # Calculate property age
    year_built = pd.to_numeric(
        data["YearBuilt"], errors="coerce"
    )

    data["PropertyAge"] = year - year_built

    # Invalid property ages become missing values
    data.loc[
        data["PropertyAge"] < 0, "PropertyAge"
    ] = np.nan

    return data


# Apply the same feature engineering to each split
train_df = add_temporal_features(train_df)
val_df = add_temporal_features(val_df)
test_df = add_temporal_features(test_df)

print(
    train_df[
        ["YearBuilt", "PropertyAge", "month_sin", "month_cos"]
    ].head()
)

        YearBuilt  PropertyAge  month_sin  month_cos
492951     1961.0         64.0       -0.5   0.866025
492952     1949.0         76.0       -0.5   0.866025
492953     1978.0         47.0       -0.5   0.866025
492954     2000.0         25.0       -0.5   0.866025
492955     1950.0         75.0       -0.5   0.866025


In [180]:
excluded_columns = {
    "ClosePrice",
    "ListPrice",
    "OriginalListPrice",
    "DaysOnMarket",
    "ListingKey",
    "ListingKeyNumeric",
    "StreetNumberNumeric",
    "AssociationFee",
    "BuyerAgencyCompensation",
    "TaxYear",
    "TaxAnnualAmount"
}

# Existing approved numeric candidates
numeric_features = [
    col for col in available_numeric_features
    if col in train_df.columns
    and col not in excluded_columns
]

# Add our three engineered features
numeric_features += [
    "PropertyAge",
    "month_sin",
    "month_cos"
]

# Remove duplicate entries
numeric_features = list(dict.fromkeys(numeric_features))

# Remove columns entirely missing in training
numeric_features = [
    col for col in numeric_features
    if not train_df[col].isna().all()
]

# Existing categorical candidates
categorical_features = [
    col for col in categorical_candidates
    if col in train_df.columns
    and col not in excluded_columns
]

features = numeric_features + categorical_features

assert target not in features
assert not set(features).intersection(excluded_columns)
assert len(features) == len(set(features))

print("Numeric:", numeric_features)
print("Categorical:", categorical_features)
print("Total selected features:", len(features))

Numeric: ['Latitude', 'Longitude', 'LivingArea', 'ParkingTotal', 'LotSizeAcres', 'YearBuilt', 'BathroomsTotalInteger', 'BuildingAreaTotal', 'BedroomsTotal', 'BelowGradeFinishedArea', 'Stories', 'LotSizeArea', 'GarageSpaces', 'LotSizeSquareFeet', 'PropertyAge', 'month_sin', 'month_cos']
Categorical: ['PostalCode', 'CountyOrParish', 'City', 'HighSchoolDistrict', 'NewConstructionYN']
Total selected features: 22


In [181]:

# Separate features and target
X_train = train_df[features].copy()
y_train = train_df[target].copy()

X_val = val_df[features].copy()
y_val = val_df[target].copy()

X_test = test_df[features].copy()
y_test = test_df[target].copy()

# Numeric preprocessing: impute missing values + scale
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical preprocessing: impute + one-hot encode
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(
        strategy="most_frequent"
    )),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        min_frequency=20
    ))
])

# Combine preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ],
    remainder="drop"
)

# Fit on training data
X_train_processed = preprocessor.fit_transform(X_train)

# 6. Transform validation and test 
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

# 7. Check output shapes
print("Train:", X_train_processed.shape)
print("Validation:", X_val_processed.shape)
print("Test:", X_test_processed.shape)

Train: (36626, 1211)
Validation: (11560, 1211)
Test: (12482, 1211)
